# Extract, Normalize and Chunk Example

In [ ]:
# Colab dependency conflicts require a workaround installing dependencies. Don't ask...
# !pip install "unstructured[pdf]" --no-deps
# !pip install emoji psutil backoff dataclasses-json filetype langdetect python-iso639 python-magic python-oxmsg python-oxmsg rapidfuzz unstructured-client

# !pip install pymupdf
# !pip install -U langchain-community langchain-text-splitters

In [ ]:
import textwrap as tw

## Load a PDF Document

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# To get rid of the deprecation warning:
# !pip install langchain_pymupdf4llm
# from langchain_pymupdf4llm import PyMuPDF4LLMLoader

In [ ]:
loader = PyMuPDFLoader("exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf")
documents = loader.load()

In [ ]:
documents[4].metadata

In [ ]:
documents[4].page_content

## Cleanup and Normalize

In [ ]:
from unstructured.cleaners.core import clean, clean_bullets, clean_extra_whitespace

def clean_text(text: str) -> str:
    
    text = clean(text)
    text = clean_bullets(text)
    text = clean_extra_whitespace(text)
    
    return text.strip()

In [ ]:
for doc in documents:
    doc.page_content = clean_text(doc.page_content)

In [ ]:
documents[4].page_content

## Chunking

#### Character-Based Recursive Chunking

The LangChain splitter class is named `RecursiveCharacterTextSplitter`. See **[here](https://gauravbytes.dev/chunking-strategies-in-langchain-rag-a-deep-dive)** to learn how it works.

The `chunk_size` and `chunk_overlap` arguments are in **characters**, not words or tokens. The average length of a word in English is 4.7 characters, and the average length of a token is $4.7 / 1.4 \sim 3.3$ characters. Therefore, if our context window is, say, $384$ tokens, then we can have chunks up to $~1200$ characters long (+ a safety margin).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = splitter.split_documents(documents[3:])

In [ ]:
len(chunks)

In [ ]:
chunks[0].metadata

In [ ]:
chunks[1].metadata

In [ ]:
len(chunks[0].page_content)

In [ ]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

#### Sentence-Based Chunking

In [ ]:
# !python -m spacy download en

In [ ]:
from langchain_text_splitters import SpacyTextSplitter

In [ ]:
splitter = SpacyTextSplitter(chunk_size=1200)  
chunks = splitter.split_documents(documents[3:])

In [ ]:
len(chunks)

In [ ]:
len(chunks[0].page_content)

In [ ]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

#### Semantic Chunking

Steps:
1. The document is first broken down into individual sentences.
2. An embedding model converts each sentence into a numerical vector representation.
3. Semantic similarity is calculated between consecutive sentence pairs.
4. Wherever the similarity score between two consecutive sentences drops significantly, the algorithm infers a shift in topic and places a chunk boundary.

LangChain's variant: "At a high level, this splits into sentences, then groups into groups of 3 sentences, and then merges one that are similar in the embedding space." (LangChain documentation)

In [ ]:
# !pip install langchain-experimental langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = SemanticChunker(
    embeddings=embedding_model,
    # See: https://python.langchain.com/docs/how_to/semantic-chunker/#breakpoints
    breakpoint_threshold_type="percentile",   # 'standard_deviation', 'interquartile', 'gradient'
    breakpoint_threshold_amount=90.0,
)

chunks = splitter.split_documents(documents[3:])

In [ ]:
len(chunks)

In [ ]:
len(chunks[0].page_content)

In [ ]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

<hr/>